In [0]:
import pyspark.sql.functions as F

catalog_name = "f1_dev"
target_table = f"{catalog_name}.silver.dim_circuits"



In [0]:
# 1. Read raw circuits table
df_circuits_raw = spark.table(f"{catalog_name}.bronze.circuits")



In [0]:
# 2. Transform & Cleanse
df_circuits_silver = (
    df_circuits_raw
    .replace(r"\N", None)
    .select(
        F.col("circuitId").cast("integer").alias("circuit_id"),
        F.col("circuitRef").alias("circuit_ref"),
        F.col("name").alias("circuit_name"),
        F.col("location"),
        F.col("country"),
        F.col("lat").alias("latitude"),
        F.col("lng").alias("longitude"),
        F.col("alt").alias("altitude")
    )
)



In [0]:
# 3. Write as Delta table in Silver schema
df_circuits_silver.write.mode("overwrite").saveAsTable(target_table)



In [0]:
# 4. Set Unity Catalog Governance Tags
spark.sql(f"""
    ALTER TABLE {target_table} SET TAGS (
        'layer' = 'silver',
        'table_type' = 'dimension',
        'contains_pii' = 'false'
    )
""")

print(f"✓ Successfully built and governed: {target_table}")